In [5]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "mypassword1"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
try:
    records = list(db.read({}))
except Exception:
    records = []

if not records:
    # Offline / unreachable DB fallback so the dashboard still works
    if not os.path.exists("aac_animals_sample.csv"):
        pd.DataFrame([
            {"name":"Lucy","animal_type":"Dog","breed":"Labrador Retriever Mix","age_upon_outcome_in_weeks":52,"sex_upon_outcome":"Intact Female","latitude":30.27,"longitude":-97.74},
            {"name":"Max","animal_type":"Dog","breed":"German Shepherd","age_upon_outcome_in_weeks":60,"sex_upon_outcome":"Intact Male","latitude":30.28,"longitude":-97.73},
            {"name":"Bella","animal_type":"Dog","breed":"Border Collie","age_upon_outcome_in_weeks":32,"sex_upon_outcome":"Intact Female","latitude":30.29,"longitude":-97.75},
            {"name":"Rex","animal_type":"Dog","breed":"Rottweiler","age_upon_outcome_in_weeks":140,"sex_upon_outcome":"Intact Male","latitude":30.30,"longitude":-97.76},
            {"name":"Goldie","animal_type":"Dog","breed":"Golden Retriever","age_upon_outcome_in_weeks":80,"sex_upon_outcome":"Intact Male","latitude":30.31,"longitude":-97.77}
        ]).to_csv("aac_animals_sample.csv", index=False)
    df = pd.read_csv("aac_animals_sample.csv")
else:
    df = pd.DataFrame.from_records(records)

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)





#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = "grazioso_logo.png" # replace with your own image
# keep the original approach safe so missing files don't crash the app
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
except Exception:
    encoded_image = None

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Hr(),


    html.Div(
        [
            # Prefer /assets logo. If you want to use the base64 inline image above,
            # uncomment the next two lines and comment out the /assets logo.
            # html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()) if encoded_image else "", style={"height": "60px"}),
            html.A(
                html.Img(src="/assets/grazioso_logo.png", style={"height": "60px"}),
                href="https://www.snhu.edu", target="_blank"),

            html.Span(" | Grazioso Salvare Dashboard – By Ryan Peguero", style={"marginLeft": "8px", "fontWeight": "bold"})
        ],
        style={"display": "flex", "alignItems": "center", "gap": "8px"}
    ),

    html.Br(),

    html.Div(
        [
            #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
            html.Label("Rescue Type Filter"),
            dcc.RadioItems(
                id="filter-type",
                options=[
                    {"label": "Reset (All)", "value": "reset"},
                    {"label": "Water Rescue", "value": "water"},
                    {"label": "Mountain/Wilderness", "value": "mountain"},
                    {"label": "Disaster/Tracking", "value": "disaster"},
                ],
                value="reset",
                inline=True
            )
        ]
    ),


    html.Hr(),


    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        # FIXME: Set up the features for your interactive data table to make it user-friendly for your client
        # If you completed the Module Six Assignment, you can copy in the code you created here
        page_size=10,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={"minWidth": 80, "maxWidth": 300, "whiteSpace": "normal"}
    ),

    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={"flex": 1}
            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={"flex": 1}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

FILTERS = {
    "water": {"breeds": ["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"], "sex": "Intact Female", "age_min": 26, "age_max": 156},
    "mountain": {"breeds": ["German Shepherd","Alaskan Malamute","Old English Sheepdog","Siberian Husky","Rottweiler"], "sex": "Intact Male", "age_min": 26, "age_max": 156},
    "disaster": {"breeds": ["Doberman Pinscher","German Shepherd","Golden Retriever","Bloodhound","Rottweiler"], "sex": "Intact Male", "age_min": 20, "age_max": 300}
}

def _apply_filter_locally(df_in, fkey):
    if fkey not in FILTERS:
        return df_in.copy()
    spec = FILTERS[fkey]
    dff = df_in.copy()
    if "breed" in dff.columns:
        dff = dff[dff["breed"].isin(spec["breeds"])]
    if "sex_upon_outcome" in dff.columns:
        dff = dff[dff["sex_upon_outcome"] == spec["sex"]]
    if "age_upon_outcome_in_weeks" in dff.columns:
        dff = dff[(dff["age_upon_outcome_in_weeks"] >= spec["age_min"]) & (dff["age_upon_outcome_in_weeks"] <= spec["age_max"])]
    return dff


# -------------------- ADDED: minimal pass-through so app runs clean --------------------
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    ## FIX ME Add code to filter interactive data table with MongoDB queries

    def _find_col(cols, candidates):
        lc = {c.lower(): c for c in cols}
        for cand in candidates:
            if cand.lower() in lc:
                return lc[cand.lower()]
        # loose match
        for c in cols:
            for cand in candidates:
                if cand.lower().replace("_","") in c.lower().replace("_",""):
                    return c
        return None

    if filter_type == "reset" or filter_type is None:
        dff = df.copy()
    else:
        specs = {
            "water":    {"breeds": ["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"], "sex": "Intact Female", "age_min": 26, "age_max": 156},
            "mountain": {"breeds": ["German Shepherd","Alaskan Malamute","Old English Sheepdog","Siberian Husky","Rottweiler"], "sex": "Intact Male",   "age_min": 26, "age_max": 156},
            "disaster": {"breeds": ["Doberman Pinscher","German Shepherd","Golden Retriever","Bloodhound","Rottweiler"],        "sex": "Intact Male",   "age_min": 20, "age_max": 300},
        }
        spec = specs.get(filter_type, None)
        dff = df.copy()
        if spec:
            breed_col = _find_col(dff.columns, ["breed"])
            sex_col   = _find_col(dff.columns, ["sex_upon_outcome","sex"])
            age_col   = _find_col(dff.columns, ["age_upon_outcome_in_weeks","age_in_weeks","ageuponoutcomeinweeks"])
            if breed_col:
                dff = dff[dff[breed_col].isin(spec["breeds"])]
            if sex_col:
                dff = dff[dff[sex_col] == spec["sex"]]
            if age_col:
                dff = dff[(dff[age_col] >= spec["age_min"]) & (dff[age_col] <= spec["age_max"])]

    if '_id' in dff.columns:
        dff = dff.drop(columns=['_id'])
    return dff.to_dict('records')



# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    if not viewData:
        return []
    dff = pd.DataFrame(viewData)
    if "breed" not in dff.columns or dff.empty:
        return []
    fig = px.pie(dff, names='breed', title='Preferred Breeds (Filtered)')
    return [dcc.Graph(figure=fig)]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if not viewData:
        return []

    if not isinstance(index, list) or len(index) == 0:
        index = [0]
    dff = pd.DataFrame.from_dict(viewData)
    row = min(index[0], len(dff) - 1)

    # Prefer named columns if present
    lat_col = next((c for c in dff.columns if c.lower() in ("latitude", "lat")), None)
    lon_col = next((c for c in dff.columns if c.lower() in ("longitude", "lon", "long")), None)
    name_col = next((c for c in dff.columns if c.lower() == "name"), None)
    breed_col = next((c for c in dff.columns if c.lower() == "breed"), None)

    if lat_col and lon_col:
        try:
            lat = float(dff.iloc[row][lat_col])
            lon = float(dff.iloc[row][lon_col])
        except Exception:
            return []
    else:
        # Fallback to starter's positional columns if needed
        if dff.shape[1] <= 14:
            return []
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]

    tooltip_text = str(dff.iloc[row][breed_col]) if breed_col else ""
    popup_name = str(dff.iloc[row][name_col]) if name_col else "Animal"

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],  # center on the selected animal
            zoom=12,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(tooltip_text),
                    dl.Popup([
                        html.H1("Animal Name"),
                        html.P(popup_name)
                    ])
                ])
            ]
        )
    ]




app.run_server(debug=True)

Read Error: nv-desktop-services.apporto.com:33900: [Errno 111] Connection refused, Timeout: 30s, Topology Description: <TopologyDescription id: 68a11534f54ad96084b893e7, topology_type: Single, servers: [<ServerDescription ('nv-desktop-services.apporto.com', 33900) server_type: Unknown, rtt: None, error=AutoReconnect('nv-desktop-services.apporto.com:33900: [Errno 111] Connection refused')>]>
Dash app running on http://127.0.0.1:12168/
